# 18 - Landmark search & preference matching

Not one of the 16 CampusX notebooks - two recommender tools added later:

- Landmark search (`src/recommender/landmark_search.py`): projects within *N* km of a named landmark.
- Preference matching (`src/recommender/listing_recommender.py`): hard filters plus a weighted closeness ranking over listings.

The item-to-item "similar developments" tool is notebook 16.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
warnings.simplefilter('ignore')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100

## 0. Data loaded

In [2]:
_ap_path = REPO_ROOT / 'data/raw/appartments.csv'
_lst_path = REPO_ROOT / 'data/processed/gurgaon_properties_missing_value_imputation.csv'
_ap_raw = pd.read_csv(_ap_path)
_lst = pd.read_csv(_lst_path)
# landmark_search / SimilarProjectsRecommender drop one embedded repeated-header row
_ap_projects = int((_ap_raw['PropertyName'].astype(str).str.strip() != 'PropertyName').sum())
print(f'Loaded {_ap_path.relative_to(REPO_ROOT)}: {len(_ap_raw)} rows '
      f'({_ap_projects} projects after dropping the embedded header row)')
print(f'Loaded {_lst_path.relative_to(REPO_ROOT)}: {len(_lst)} rows')

Loaded data\raw\appartments.csv: 247 rows (246 projects after dropping the embedded header row)
Loaded data\processed\gurgaon_properties_missing_value_imputation.csv: 3554 rows


---
## 1. Landmark-proximity search

`build_distance_matrix()` parses every `LocationAdvantages` blob into distance-to-landmark in metres (`distance_to_metres`, the corrected parser). `search_by_landmark(name, radius_km)` returns the projects inside that radius, nearest first.

In [3]:
from src.recommender.landmark_search import (
    build_distance_matrix, search_by_landmark, list_landmarks,
)

dist = build_distance_matrix(save=False)   # don't overwrite the committed matrix
no_data = int((~dist.notna().any(axis=1)).sum())
print(f'{dist.shape[0]} projects x {dist.shape[1]} landmarks | '
      f'{int(dist.notna().sum().sum())} known distances | '
      f'{no_data} projects with no parseable landmark distance (all-NaN rows)')
print('landmarks listed by >= 10 projects:', len(list_landmarks(dist, min_projects=10)))

246 projects x 965 landmarks | 2370 known distances | 17 projects with no parseable landmark distance (all-NaN rows)
landmarks listed by >= 10 projects: 30


In [4]:
for lm, radius in [('Dwarka Expressway', 2), ('Sector 55-56 Metro Station', 5),
                   ('Indira Gandhi International Airport', 18)]:
    hits = search_by_landmark(lm, radius, matrix=dist)
    print(f'\n{lm} - within {radius} km: {len(hits)} projects')
    print(hits.head(6).to_string(index=False))


Dwarka Expressway - within 2 km: 21 projects
               project  distance_m  distance_km
           M3M Capital          10         0.01
   Satya Merano Greens          75         0.08
     Indiabulls Enigma         100         0.10
Puri Diplomatic Greens         300         0.30
      Puri Emerald Bay         350         0.35
             M3M Crown         450         0.45

Sector 55-56 Metro Station - within 5 km: 10 projects
               project  distance_m  distance_km
    Puri The Aravallis        2700          2.7
     Mahindra Luminare        2700          2.7
     Anant Raj Estates        3600          3.6
Anant Raj Ashok Estate        3700          3.7
   Adani Samsara Avasa        3800          3.8
       Emaar Digihomes        3900          3.9

Indira Gandhi International Airport - within 18 km: 26 projects
                 project  distance_m  distance_km
      Ambience Creacions       11000         11.0
               M3M Crown       14100         14.1
Emaar MGF Em

In [5]:
# edge cases
empty = search_by_landmark('Sector 55-56 Metro Station', 2, matrix=dist)  # nearest is 2.7 km
print('no match within radius -> empty frame, normal columns:',
      empty.empty, list(empty.columns))

for bad in (0, -3):
    try:
        search_by_landmark('Dwarka Expressway', bad, matrix=dist)
    except ValueError as e:
        print(f'radius_km={bad} -> ValueError: {e}')

try:
    search_by_landmark('Buckingham Palace', 5, matrix=dist)
except KeyError as e:
    print('unknown landmark ->', str(e)[:90], '...')

no match within radius -> empty frame, normal columns: True ['project', 'distance_m', 'distance_km']
radius_km=0 -> ValueError: radius_km must be > 0, got 0
radius_km=-3 -> ValueError: radius_km must be > 0, got -3
unknown landmark -> "landmark not found: 'Buckingham Palace'. Did you mean: ['Neemrana Palace', 'Hyatt Place', ...


**Reading it.** `Dwarka Expressway` @ 2 km returns 21 projects, several essentially on the road (M3M Capital 10 m, Satya Merano Greens 75 m). The metro station's nearest project is 2.7 km, so a 2 km search correctly returns nothing - an **empty DataFrame with the normal columns** (`project`, `distance_m`, `distance_km`), not an error and not a silent `None`. `radius_km <= 0` raises `ValueError` (km in, never metres - no off-by-1000). `IGI Airport` folds to `Indira Gandhi International Airport` via a small alias map; the long tail of name variants is not resolved.

---
## 2. Preference-based listing recommender

Takes a **preference vector** over individual listings (`gurgaon_properties_missing_value_imputation.csv`), applies **hard filters** (budget ceiling, min bedrooms, sector, property type), then ranks the survivors by **weighted closeness** on the soft axes the user specified: `built_up_area` 0.35 / `luxury_score` 0.25 / `bathroom` 0.15 / `agePossession` 0.15 / `furnishing_type` 0.10, each scaled by its own IQR-or-std over the full table. The price model and project-similarity are **display columns only**, never in the ranking score. Design + the two deliberate scope choices (luxury weight vs price-importance; sector as a hard filter only): `reports/recommender/listing_recommender_design.md`.

In [6]:
from src.recommender.listing_recommender import ListingRecommender, Preferences

listings = ListingRecommender.from_csv()
print('axis scales:', {k: round(v, 1) for k, v in listings._scale.items()})
print('society -> project bridge:', len(listings._society_to_project), 'societies')

VIEW = ['society', 'sector', 'property_type', 'price', 'bedRoom', 'bathroom',
        'built_up_area', 'agePossession', 'furnishing_type', 'match_score',
        'predicted_price_cr', 'price_vs_model_pct', 'similar_projects']

axis scales: {'area_sqft': 997.5, 'luxury_score': 77.0, 'bathrooms': 1.5, 'age_possession': 1.0, 'furnishing': 0.6}
society -> project bridge: 172 societies


### A - 3+ BHK flat, <= Rs 2.5 Cr, ~1600 sqft, semi-furnished, Relatively New

In [7]:
a = listings.recommend(Preferences(
    budget_max_cr=2.5, min_bedrooms=3, property_type='flat',
    area_sqft=1600, furnishing='semifurnished', age_possession='Relatively New',
), k=6)
a[VIEW]

,society,sector,property_type,price,bedRoom,bathroom,built_up_area,agePossession,furnishing_type,match_score,predicted_price_cr,price_vs_model_pct,similar_projects
0,emaar mgf emerald floors premier,sector 65,flat,2.25,3.0,3.0,1600.0,Relatively New,1.0,0.0,2.32,-3.0,"Adani Brahma Samsara, Optimal ultra luxury bui..."
1,vatika gurgaon,sector 83,flat,1.10,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.21,-9.1,None
2,emaar palm gardens,sector 83,flat,1.75,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.73,1.2,"Tulip Yellow, Orchid Petals, Adani M2K Oyster ..."
3,emaar palm gardens,sector 83,flat,1.72,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.73,-0.6,"Tulip Yellow, Orchid Petals, Adani M2K Oyster ..."
4,emaar mgf emerald floors premier,sector 65,flat,2.40,3.0,3.0,1600.0,Relatively New,1.0,0.0,2.32,3.4,"Adani Brahma Samsara, Optimal ultra luxury bui..."
5,emaar palm gardens,sector 83,flat,1.75,3.0,3.0,1600.0,Relatively New,1.0,0.0,1.73,1.2,"Tulip Yellow, Orchid Petals, Adani M2K Oyster ..."


### B - 2+ BHK, <= Rs 1 Cr, in sector 92, ~1200 sqft (hard-filter heavy)

In [8]:
b = listings.recommend(Preferences(
    budget_max_cr=1.0, min_bedrooms=2, sector='sector 92', area_sqft=1200,
), k=6)
b[VIEW]

,society,sector,property_type,price,bedRoom,bathroom,built_up_area,agePossession,furnishing_type,match_score,predicted_price_cr,price_vs_model_pct,similar_projects
0,ansal heights,sector 92,flat,0.60,2.0,2.0,1195.00,Under Construction,0.0,0.005013,0.75,-20.0,None
1,parkwood westend,sector 92,flat,0.70,2.0,2.0,1217.00,Under Construction,0.0,0.017043,0.78,-10.3,None
2,sare homes,sector 92,flat,0.82,3.0,3.0,1221.18,Relatively New,0.0,0.021233,0.82,0.0,None
3,sare crescent parc royal greens phase 1,sector 92,flat,0.79,3.0,2.0,1175.00,New Property,0.0,0.025063,0.80,-1.2,None
4,sare green parc phase 3,sector 92,flat,0.79,3.0,2.0,1175.00,New Property,0.0,0.025063,0.80,-1.2,None
5,raheja sampada,sector 92,flat,0.65,3.0,2.0,1240.00,Relatively New,0.0,0.040100,0.86,-24.4,None


### C - no match: 4+ BHK flat under Rs 0.5 Cr

In [9]:
c_res = listings.recommend(Preferences(
    budget_max_cr=0.5, min_bedrooms=4, property_type='flat',
), k=6)
print('empty:', c_res.empty, '| rows:', len(c_res), '| columns:', list(c_res.columns))

empty: True | rows: 0 | columns: ['society', 'sector', 'property_type', 'price', 'price_per_sqft', 'bedRoom', 'bathroom', 'built_up_area', 'agePossession', 'furnishing_type', 'floorNum', 'luxury_score', 'match_score', 'predicted_price_cr', 'price_vs_model_pct', 'similar_projects']


### D - hard filters only, no soft axis: `match_score` is NaN, sorted by price

In [10]:
d = listings.recommend(Preferences(budget_max_cr=1.2, min_bedrooms=2), k=5)
print('match_score all NaN:', d['match_score'].isna().all())
d[VIEW]

match_score all NaN: True


,society,sector,property_type,price,bedRoom,bathroom,built_up_area,agePossession,furnishing_type,match_score,predicted_price_cr,price_vs_model_pct,similar_projects
0,ashiana apartment,sector 23,flat,0.16,2.0,2.0,706.0,Moderately Old,0.0,NaN,0.35,-54.3,None
1,hcbs sports ville,sohna road,flat,0.20,2.0,2.0,743.0,New Property,0.0,NaN,0.28,-28.6,None
2,vidya apartment,sector 5,flat,0.22,2.0,2.0,570.0,Relatively New,2.0,NaN,0.30,-26.7,None
3,independent,sector 9,house,0.22,2.0,2.0,37.0,Old Property,0.0,NaN,0.46,-52.2,None
4,city shri ram apartments 1,sector 110,flat,0.22,2.0,1.0,543.0,Relatively New,0.0,NaN,0.29,-24.1,None


**Reading it.** (A) every result is a 3 BHK / ~1600 sqft / Relatively New / semi-furnished flat - `match_score` 0 means the specified axes matched exactly; `price_vs_model_pct` shows each priced within a few percent of the model. (B) the `sector 92` hard filter binds; `area_sqft` is the only soft axis, so ranking is by area closeness. (C) an impossible combination returns an **empty DataFrame with the full column set**, not an error. (D) with no soft preference, `match_score` is `NaN` and results come back cheapest-first. Note the source table has near-duplicate listings (same unit type, slightly different price) - not de-duplicated in v1 (`listing_recommender_design.md`).

---
## Recommender features - status

| Feature | Module | Notebook |
|---|---|---|
| Similar developments (item-to-item) | `SimilarProjectsRecommender` | `16_recommender_system` (+ blend-resolution section) |
| Landmark-proximity search | `landmark_search` | this notebook, Section 1 |
| Preference-based listing recommender | `ListingRecommender` | this notebook, Section 2 |

All three feed the app's Recommendations page.